# Enhancing Recruitment with Prompt Engineering using Python and Google Gemini AI
This notebook demonstrates how to use Google's Gemini models and prompt engineering to enhance various tasks in recruitment, such as generating job descriptions, screening resumes, and generating interview questions.

In [11]:
# Install Google GenAI SDK if not already installed
!pip install --upgrade google-genai

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# Import required libraries
from google import genai
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain prompt engineering in simple words."
)

print(response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


**Prompt engineering** is simply the art of **asking an AI the right question to get the best possible answer.**

Think of an AI (like ChatGPT) as a **super-smart assistant who can’t read your mind.** 

* If you give it vague instructions, you’ll get a generic or messy answer. 
* If you give it clear, detailed instructions, you’ll get a brilliant answer.

Prompt engineering is just learning how to give those clear, detailed instructions.

---

### A Simple Example

Imagine you want a recipe.

* **A Bad Prompt:** "Give me a recipe for dinner."  
  *(The AI might give you a complex 3-hour French dish when you only had 15 minutes and a block of cheese.)*

* **An "Engineered" Prompt:** "Give me a quick 15-minute dinner recipe using chicken and rice. Make it easy for a beginner, and list the ingredients first."  
  *(Now, the AI knows exactly what you need and gives you the perfect answer.)*

---

### The 4 Secrets of a Good Prompt

To "engineer" a great prompt, you usually just add four th

In [3]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

## 1. Job Description Generator

In [4]:
# Function to generate job description based on role
from google import genai

def generate_job_description(role):
    prompt = f"Write a detailed job description for the role of {role} in a tech company."
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

# Example usage
print(generate_job_description("Data Engineer"))

Here is a comprehensive and modern job description for a **Data Engineer** at a tech company. You can customize the bracketed information **[like this]** to match your company's specific needs.

---

# Job Description: Data Engineer

**Position:** Data Engineer  
**Department:** Data & Engineering  
**Location:** [Remote / Hybrid / Onsite - City, State]  
**Employment Type:** Full-time  
**Experience Level:** Mid–Senior Level  

---

### **About [Company Name]**
At **[Company Name]**, we are building the future of **[Industry/Product focus, e.g., FinTech, SaaS, E-commerce]**. We process millions of events daily and rely heavily on data to make product decisions, optimize customer experiences, and drive business growth. 

We are looking for a talented **Data Engineer** to help us build, scale, and optimize our next-generation data platform. If you thrive on solving complex distributed systems problems, building resilient data pipelines, and turning raw data into actionable insight, we’d

## 2. Resume Screening and Matching Candidates to JD

In [5]:
!pip install google-genai PyMuPDF python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [6]:
# Function to screen resume against job requirements
def screen_resume(resume_text, job_requirements):
    prompt = (
        f"Given the following job requirements: {job_requirements}\n"
        f"And the following resume: {resume_text}\n"
        "Evaluate the match on a scale of 1 to 10 and explain why."
    )
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

# Example usage
resume = "Experienced software engineer with Python and cloud experience."
requirements = "5+ years experience, strong Python skills, cloud deployment knowledge"
print(screen_resume(resume, requirements))

**Match Score: 4.5 / 10**

### **Explanation:**

While the resume mentions keywords that align with all three requirements, it lacks the depth, metrics, and specific details necessary to confirm that the candidate actually meets the criteria. 

Here is the detailed breakdown:

1. **5+ Years Experience (Score: Unclear / Low Match)**
   * **Job Need:** A minimum of 5 years of professional experience.
   * **Resume:** States "Experienced software engineer." 
   * **Gap:** "Experienced" is subjective. Without dates, a timeline, or job history, it is impossible to verify if the candidate meets the 5-year minimum threshold. 

2. **Strong Python Skills (Score: Partial Match)**
   * **Job Need:** *Strong* Python skills.
   * **Resume:** Mentions "Python experience."
   * **Gap:** The resume acknowledges knowing Python, but gives no evidence of "strong" skills (e.g., specific frameworks like Django/FastAPI, libraries, data structures, or complex projects built).

3. **Cloud Deployment Knowledge

In [7]:
!pip install pytesseract pillow
!sudo apt-get install tesseract-ocr
import pytesseract
from PIL import Image
import fitz
RESUME_FOLDER = "./resumes"
os.makedirs(RESUME_FOLDER, exist_ok=True)

# Function to extract text from PDF
def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        # Try text first
        page_text = page.get_text()
        if not page_text.strip():
            # Fallback to OCR
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            page_text = pytesseract.image_to_string(img)
        text += page_text
    return text.strip()


Defaulting to user installation because normal site-packages is not writeable


'sudo' is not recognized as an internal or external command,
operable program or batch file.


In [8]:

# Process each resume
for filename in os.listdir(RESUME_FOLDER):
    if filename.endswith(".pdf"):
        file_path = os.path.join(RESUME_FOLDER, filename)
        print(f"\n📄 Analyzing {filename}...\n{'='*50}")
        resume_text = extract_text_from_pdf(file_path)
        print(resume_text)
        result = screen_resume(resume_text, requirements)
        print(result)
        print("\n" + "-"*80)

## 4. Generating Interview Questions

In [9]:
# Function to generate interview questions
def generate_interview_questions(role):
    prompt = f"Generate 5 interview questions for the role of {role}."
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text

# Example usage
print(generate_interview_questions("Machine Learning Engineer"))

Here are 5 interview questions for a Machine Learning Engineer role, covering core ML theory, system design, data engineering, MLOps, and real-world troubleshooting.

---

### 1. ML System Design (Real-Time vs. Batch)
> **"Design a real-time personalized recommendation system for an e-commerce platform (e.g., Amazon). How would you structure the pipeline from data ingestion to model serving, and how do you handle low-latency constraints?"**

* **Why ask this:** Tests the candidate’s ability to build end-to-end ML architectures, balance trade-offs (e.g., accuracy vs. latency), and understand infrastructure beyond just writing model code.
* **What to look for:**
  * Division of the system into **Retrieval (Candidate Generation)** and **Ranking** stages.
  * Use of feature stores and vector databases (e.g., FAISS, Milvus).
  * Strategy for real-time features (streaming via Kafka/Flink) vs. batch features.
  * Strategies to achieve sub-100ms response times (e.g., model quantization, cachin

## Conclusion
This notebook showcased how prompt engineering and Google Gemini APIs can automate and enhance critical recruitment tasks like job description writing, resume evaluation, and candidate matching. In production, integrate with HR systems and databases for streamlined workflows.